# Export des tables Gold vers PostgreSQL

Exécuter après `03_gold.ipynb`. Chaque table PostgreSQL `public.gold_*` ci-dessous est remplacée par la table correspondante du data lake. Les autres tables ne sont pas modifiées. Les nombres de lignes sont vérifiés après relecture.

In [ ]:
import sys
from pathlib import Path

racine = Path.cwd()
if not (racine / 'commun').is_dir():
    racine = racine.parent
if not (racine / 'commun').is_dir():
    raise FileNotFoundError('Exécuter ce notebook depuis le dossier work du projet.')
sys.path.insert(0, str(racine))
from commun.spark_utils import LAKE, PG_PROPS, PG_URL, creer_session, format_tables, vers_postgres

spark = creer_session('Export_Gold_PostgreSQL', lac=True, delta=True, postgres=True)
noms = ['gold_sites_classement', 'gold_sites_mensuel', 'gold_serie_mensuelle', 'gold_serie_quotidienne', 'gold_heatmap_heure_jour', 'gold_kpi']
for nom in noms:
    table = spark.read.format(format_tables(spark)).load(f'{LAKE}/gold/{nom}')
    attendu = table.count()
    if attendu == 0:
        raise ValueError(f'{nom} est vide : export interrompu.')
    vers_postgres(table, nom, mode='overwrite')
    obtenu = spark.read.jdbc(PG_URL, nom, properties=PG_PROPS).count()
    if obtenu != attendu:
        raise RuntimeError(f'{nom} : {obtenu} lignes dans PostgreSQL, {attendu} attendues.')
    print(f'{nom} : {obtenu} lignes vérifiées dans PostgreSQL.')